[Reference](https://medium.com/mitb-for-all/how-to-train-your-llm-teaching-toothless-to-bite-8d9f56fe4b2a)

In [24]:
# pip install llama_index

In [25]:
# pip install llama-index-llms-ollama

In [27]:
# ./notebooks/training_dataset_gen.ipynb

import nest_asyncio
nest_asyncio.apply()

from llama_index.core import SimpleDirectoryReader
from llama_index.core.llama_dataset.generator import RagDatasetGenerator
from llama_index.llms.ollama import Ollama

!mkdir  -p ../data
!wget "https://arxiv.org/pdf/2405.00247.pdf" -O "../data/non_traditional_credentials.pdf"

docs = SimpleDirectoryReader("../data/").load_data(show_progress=True)

In [22]:
# !pip install colab-xterm
# %load_ext colabxterm

In [23]:
# %xterm

```
$ curl https://ollama.ai/install.sh | sh
$ ollama serve &
$ ollama pull mistral
```

In [26]:
# %pip install -U langchain-ollama

In [ ]:
!ollama list

NAME              ID              SIZE      MODIFIED           
mistral:latest    f974a74358d6    4.1 GB    About a minute ago    


In [ ]:
!ollama - version

Error: unknown command "version" for "ollama"


In [ ]:
!ollama pull llama
!ollama generate "Hello, world!"


Error: pull model manifest: file does not exist
Error: unknown command "generate" for "ollama"


In [ ]:
data_gen = RagDatasetGenerator.from_documents(
    docs,
    llm= Ollama("mistral"),
    question_gen_query="You are a teacher/professor. Using the provided context, formulat a single question and its answer",
    num_questions_per_chunk=10
)
qa_dataset = data_gen.generate_dataset_from_nodes()

In [ ]:
from sklearn.model_selection import train_test_split
from llama_index.core.llama_dataset import LabelledRagDataset
import json
import pandas as pd

all_examples = qa_dataset.examples
train_examples, test_examples = train_test_split(
    all_examples,
    test_size=0.2,          # 20% held out
    random_state=42,        # for reproducibility
    shuffle=True
)
print(f"Training on {len(train_examples)} examples, testing on {len(test_examples)} examples")

training_dataset = LabelledRagDataset(examples=train_examples)
holdout_dataset = LabelledRagDataset(examples=test_examples)

records = []
for ex in holdout_dataset.examples:
    records.append({
        "query": ex.query,
        # JSON-encode the list of contexts
        "reference_contexts": json.dumps(ex.reference_contexts),
        "reference_answer": ex.reference_answer,
        # JSON-encode the CreatedBy objects
        "query_by": ex.query_by.model_dump_json(),
        "reference_answer_by": ex.reference_answer_by.model_dump_json(),
    })

df = pd.DataFrame.from_records(records)
df.to_csv("holdout_dataset.csv", index=False)

print(f"Training on {len(train_examples)} examples, testing on {len(test_examples)} examples")

In [ ]:
def serialize_to_jsonl(examples, out_path="train.jsonl"):
    """
    examples: list of LabelledRagDataExample,
              each with .query (str) and .reference_answer (str)
    out_path:  path to write the JSONL file
    """
    def strip_prefix(text):
        # remove leading **Question:** or **Answer:** if present
        for p in ("**Question:**", "**Answer:**"):
            if text.strip().startswith(p):
                return text.strip()[len(p):].strip()
        return text

    with open(out_path, "w", encoding="utf8") as f:
        for ex in examples:
            q_raw = ex.query or ""
            a_raw = getattr(ex, "reference_answer", None)
            # only serialize if this is a 'Question' example and has an answer
            if q_raw.lower().startswith("**question") and a_raw:
                q = strip_prefix(q_raw)
                a = a_raw.strip()
                obj = {
                    "messages": [
                        {"role": "user",      "content": q},
                        {"role": "assistant", "content": a}
                    ]
                }
                f.write(json.dumps(obj, ensure_ascii=False) + "\n")

serialize_to_jsonl(train_examples)

In [ ]:
from llama_index.core.llama_dataset import (
    LabelledRagDataset,
    LabelledRagDataExample,
    CreatedBy,
)

def get_rag_dataset_from_csv(csv_path: str):
    converters = {
        "reference_contexts":    lambda s: json.loads(s),
        "query_by":             lambda s: CreatedBy.model_validate_json(s),
        "reference_answer_by":  lambda s: CreatedBy.model_validate_json(s),
    }
    df = pd.read_csv(csv_path, converters=converters)
    examples = []
    for _, row in df.iterrows():
        examples.append(
            LabelledRagDataExample(
                query=row["query"],
                query_by=row["query_by"],                      # now a CreatedBy
                reference_contexts=row["reference_contexts"],   # now a List[str]
                reference_answer=row["reference_answer"],
                reference_answer_by=row["reference_answer_by"], # now a CreatedBy
            )
        )

    # 4. Create the dataset
    dataset = LabelledRagDataset(examples=examples)
    return dataset

holdout_dataset = get_rag_dataset_from_csv("holdout_dataset.csv")

In [ ]:
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama
from llama_index.core import VectorStoreIndex

embed_model = OllamaEmbedding(model_name="nomic-embed-text")
index = VectorStoreIndex.from_documents(docs, embed_model=embed_model)
query_engine = index.as_query_engine(
      similarity_top_k=6,
      llm = Ollama("llama3.2:1b") # this is the LLM we want to finetune
)

In [ ]:
from llama_index.core.llama_pack import download_llama_pack

RagEvaluatorPack = download_llama_pack("RagEvaluatorPack", "./pack")
rag_evaluator = RagEvaluatorPack(
    query_engine=query_engine,
    rag_dataset=holdout_dataset,
    judge_llm=Ollama("qwen2.5", request_timeout=120.0), #use the same llm that we use to create the dataset to judge
    embed_model=OllamaEmbedding(model_name="nomic-embed-text")
)
benchmark_df = rag_evaluator.run()

In [ ]:
# ./notebooks/finetune_llama32_1bn.ipynb

import json
from datasets import Dataset
from unsloth import FastLanguageModel
import torch

# Load datasets
def load_messages_with_system(path, system_content="You are a helpful assistant."):
    examples = []
    with open(path, 'r', encoding='utf8') as f:
        for line in f:
            obj = json.loads(line)
            sys_msg = {"role": "system", "content": system_content}
            ua_msgs = obj.get("messages", [])
            examples.append({"messages": [sys_msg] + ua_msgs})
    return examples

examples = load_messages_with_system("train.jsonl")
dataset = Dataset.from_list(examples)

# Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # Typical values: 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,  # Set to 0 for optimal performance
    bias = "none",     # "none" is the most memory-efficient option
    use_gradient_checkpointing = "unsloth",  # Use "unsloth" for 30% less VRAM usage
    random_state = 3407,
    use_rslora = False,   # Rank-stabilized LoRA support available
    loftq_config = None,  # LoftQ support is also available
)

def formatting_prompts_func(batch):
    convos = batch["messages"]           # list of message-lists
    texts = [
        tokenizer.apply_chat_template(convo,
                                      tokenize=False,
                                      add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

# 4. Map WITHOUT removing 'messages'
dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)

import os
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        max_steps = 60, # can also comment this out and set num_epochs=1
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 2025,
        report_to = "none", # Use this for WandB etc
    ),
)
trainer_stats = trainer.train()

# Push the model to HuggingFace Hub
model.push_to_hub(
  "tituslhy/retrained_llama32-1bn-finetuned",
  token = os.environ["HUGGINGFACE_ACCESS_TOKEN"]
)
tokenizer.push_to_hub(
  "tituslhy/retrained_llama32-1bn-finetuned",
  token = os.environ["HUGGINGFACE_ACCESS_TOKEN"]
)

model.push_to_hub_gguf(
    "tituslhy/retrained_llama32-1bn-finetuned",
    tokenizer,
    quantization_method = ["q4_k_m", "q8_0", "q5_k_m",], # the quantization methods
    token = os.environ["HUGGINGFACE_ACCESS_TOKEN"],
)

In [ ]:
# ./notebooks/finetune_llama32_1bn.ipynb

# Exactly as before, just with a new LLM
query_engine2 = index.as_query_engine(
    similarity_top_k=6,
    llm = Ollama("hf.co/tituslhy/retrained_llama32-1bn-finetuned:Q4_K_M")
)
rag_evaluator2 = RagEvaluatorPack(
    query_engine=query_engine2,
    rag_dataset=holdout_dataset,
    judge_llm=Ollama("qwen2.5", request_timeout=120.0), #use the same llm that we use to create the dataset to judge
    embed_model=OllamaEmbedding(model_name="nomic-embed-text")
)
benchmark_df = await rag_evaluator.arun()

from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.core.evaluation import SemanticSimilarityEvaluator
from tqdm import tqdm

llm = Ollama("hf.co/tituslhy/retrained_llama32-1bn-finetuned:Q4_K_M")
embed_model = OllamaEmbedding(model_name="nomic-embed-text")
converters = {
        "reference_contexts":   lambda s: json.loads(s),
        "query_by":             lambda s: CreatedBy.model_validate_json(s),
        "reference_answer_by":  lambda s: CreatedBy.model_validate_json(s),
    }
df = pd.read_csv("holdout_dataset.csv", converters=converters)

evaluator = SemanticSimilarityEvaluator(
    similarity_threshold=0.5,
    embed_model=embed_model
)

# Filter out proper questions
queries = [df.iloc[i]['query'] for i in range(len(df)) if "Question" in df.iloc[i]['query']]
references = [df.iloc[i]['reference_answer'] for i in range(len(df)) if "Question" in df.iloc[i]['query']]
df_answers = pd.DataFrame({'queries': queries, 'reference_answers': references})

answers, similarity_scores = [], []
for idx, (query, reference) in tqdm(df_answers.iterrows()):
    answer = llm.complete(query)
    answers.append(str(answer))

    similarity_score = evaluator.evaluate(
        response = str(answer),
        reference = reference
    )
    similarity_scores.append(round(similarity_score.score, 2))

df_answers['answers'] = answers
df_answers['similarity_scores'] = similarity_scores